In [1]:
import pandas as pd
import geopandas as gpd
from matplotlib import pyplot as plt
import numpy as np
import os
from config.config import DATA_RAW, OUTPUTS, DATA_INTERIM
from utils import *

## Land Registry Price Paid data (2022-2024)

Three years of Land Registry Price Paid data downloaded from
https://www.gov.uk/government/collections/price-paid-data.
Category A transactions only (standard residential market sales),
residential property types D/S/T/F only, null postcodes excluded.
No upper price ceiling applied — high value transactions confirmed as
genuine super-prime London residential sales.

In [2]:
land_reg_files = sorted((DATA_RAW / "land_registry").glob("*.csv"))
print(f"Files found: {[f.name for f in land_reg_files]}")

land_reg_full = pd.concat([pd.read_csv(f) for f in land_reg_files], ignore_index=True)

Files found: ['pp-2022.csv', 'pp-2023.csv', 'pp-2024.csv']


## Non-geographic filtering

In [3]:
land_reg_full = land_reg_full[[
    "transaction_unique_identifier", "price", "date_of_transfer", "postcode", "property_type", "ppd_cat_type"
]]


In [4]:
land_reg_clean = land_reg_full[
    (land_reg_full["property_type"].isin(["D", "S", "T", "F"])) &
    (land_reg_full["ppd_cat_type"] == "A") &
    (land_reg_full["postcode"].notna())
    ]

## Standardise postcodes

In [5]:
postcode_dir_full = pd.read_csv(
    DATA_RAW / "boundaries" / "ONSPD_FEB_2024_UK.csv",
    usecols=["pcds", "lsoa21"],
    encoding="latin-1",
    low_memory=False
)


In [6]:
# Standardise Land Registry postcodes to match pcds format
land_reg_clean["postcode"] = (land_reg_clean["postcode"]
                              .str.strip()
                              .str.replace(" ", "", regex=False)
                              .str.upper()
                              .apply(lambda x: f"{x[:-3]} {x[-3:]}" if pd.notna(x) else x))

# Validate against UK postcode regex
postcode_pattern = r'^[A-Z]{1,2}[0-9][0-9A-Z]?\s[0-9][A-Z]{2}$'
valid_mask = land_reg_clean["postcode"].str.match(postcode_pattern, na=False)

print(f"Valid postcodes: {valid_mask.sum()} ({valid_mask.sum()/len(land_reg_clean)*100:.2f}%)")
print(f"Invalid postcodes: {(~valid_mask).sum()}  ({(~valid_mask).sum()/len(land_reg_clean)*100:.2f}%)")

print(f"\nSample of invalid postcodes:")
print(land_reg_clean[~valid_mask]["postcode"].head(20).tolist())

print(f"\nSample of standardised postcodes:")
print(land_reg_clean["postcode"].head(20).tolist())

Valid postcodes: 2370281 (100.00%)
Invalid postcodes: 0  (0.00%)

Sample of invalid postcodes:
[]

Sample of standardised postcodes:
['DL9 4RS', 'YO12 7ND', 'HG5 0TT', 'YO31 1BU', 'LA2 7EB', 'YO21 3JJ', 'HG2 0FD', 'HG5 0BD', 'YO19 6QG', 'HG1 2JJ', 'YO26 5NF', 'DL10 4AX', 'HG3 1BQ', 'HG3 3SJ', 'YO62 4EL', 'YO8 4HH', 'YO8 5JE', 'YO21 3NF', 'YO13 9HL', 'YO22 4DY']


## Join with ONS Postcode Directory to get LSOA codes

In [7]:
land_reg_lsoa = land_reg_clean.merge(
    postcode_dir_full,
    left_on="postcode",
    right_on="pcds",
    how="left"
)

print(f"Rows before join: {len(land_reg_clean)}")
print(f"Rows after join: {len(land_reg_lsoa)}")
print(f"Null lsoa21 after join: {land_reg_lsoa['lsoa21'].isna().sum()} "
      f"({land_reg_lsoa['lsoa21'].isna().sum()/len(land_reg_lsoa)*100:.2f}%)")


Rows before join: 2370281
Rows after join: 2370281
Null lsoa21 after join: 14641 (0.62%)


In [8]:
# Drop unmatched transactions
land_reg_lsoa = land_reg_lsoa[land_reg_lsoa["lsoa21"].notna()].copy()
print(f"Rows after dropping unmatched: {len(land_reg_lsoa)}")

Rows after dropping unmatched: 2355640


## Filter to London

In [9]:
london_lsoas = gpd.read_file(
    DATA_INTERIM / "boundaries" / "london_lsoa_2021.gpkg",
    layer="london_lsoa_2021"
)[["LSOA21CD"]]

land_reg_london = land_reg_lsoa[land_reg_lsoa["lsoa21"].isin(london_lsoas["LSOA21CD"])].copy()

print(f"Rows after filtering to London: {len(land_reg_london)}")
print(f"Unique London LSOAs with transactions: {land_reg_london['lsoa21'].nunique()}")
print(f"Total London LSOAs: {len(london_lsoas)}")

Rows after filtering to London: 256991
Unique London LSOAs with transactions: 4992
Total London LSOAs: 4994


In [10]:
os.makedirs(DATA_INTERIM / "land_registry", exist_ok=True)

land_reg_london.to_csv(
    DATA_INTERIM / "land_registry" / "land_reg_london_2022_2024.csv",
    index=False
)

print(f"Saved {len(land_reg_london)} London transactions")

Saved 256991 London transactions


## Check distribution of property prices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Top left - full distribution (log scale x axis for clarity)
axes[0, 0].hist(land_reg_london["price"], bins="auto", edgecolor="black", color="steelblue")
axes[0, 0].set_title("Full distribution")
axes[0, 0].set_xlabel("Price (£)")
axes[0, 0].set_ylabel("Count")

# Top right - log scale to see the full range clearly
axes[0, 1].hist(np.log10(land_reg_london["price"]), bins="auto", edgecolor="black", color="steelblue")
axes[0, 1].set_title("Distribution (log10 scale)")
axes[0, 1].set_xlabel("log10(Price)")
axes[0, 1].set_ylabel("Count")

# Bottom left - transactions per LSOA distribution
transactions_per_lsoa = land_reg_london.groupby("lsoa21").size()
axes[1, 0].hist(transactions_per_lsoa, bins="auto", edgecolor="black", color="steelblue")
axes[1, 0].set_title("Transactions per LSOA (2022-2024)")
axes[1, 0].set_xlabel("Number of transactions")
axes[1, 0].set_ylabel("Count of LSOAs")
axes[1, 0].axvline(x=30, color="orange", linestyle="--", label="30 transaction threshold (applied)")
axes[1, 0].axvline(x=50, color="red", linestyle="dashed", label="50 transaction threshold (original paper)")
axes[1, 0].legend()

# Bottom right - median price per LSOA distribution
median_price_per_lsoa = land_reg_london.groupby("lsoa21")["price"].median()
axes[1, 1].hist(median_price_per_lsoa, bins="auto", edgecolor="black", color="steelblue")
axes[1, 1].set_title("Median transaction price per LSOA")
axes[1, 1].set_xlabel("Median transaction (£)")
axes[1, 1].set_ylabel("Count of LSOAs")

plt.tight_layout()
plt.savefig(OUTPUTS / "figures" / "london_land_reg_distribution_detail.png",
            dpi=150, bbox_inches="tight")
plt.show()

# Summary statistics
print("Transactions per LSOA: ")
print(transactions_per_lsoa.describe())
print(f"\nLSOAs below 50 transaction threshold: {(transactions_per_lsoa < 50).sum()}")
print(f"LSOAs at or above threshold: {(transactions_per_lsoa >= 50).sum()}")
print(f"\nMedian price per LSOA summary:")
print(median_price_per_lsoa.describe())

## Gini estimator stability analysis
The original paper applied a minimum of 50 property observations per LSOA,
based on Zoopla estimated values with a mean of 516 observations per LSOA.
Applying this threshold to Land Registry transaction data would systematically
exclude high-renter boroughs — notably Newham (73% of LSOAs excluded at n=50)
and Brent (71%) — which are among the most deprived and heavily policed areas
in London, introducing a systematic geographic bias into the analysis.

To determine a defensible lower threshold, a bootstrap simulation was run
on two representative LSOAs — one with low housing Gini (0.153) and one with
high housing Gini (0.349) — subsampling at sizes from n=10 to n=100 across
1,000 iterations each, measuring both the coefficient of variation (stability)
and mean estimated Gini (bias).

In [ ]:
# Take a large well-observed LSOA and repeatedly subsample it
# to see how much the Gini varies at different n

# Find a well-observed LSOA to use as the population
well_observed = transactions_per_lsoa[transactions_per_lsoa > 200].index[0]
population = land_reg_london[land_reg_london["lsoa21"] == well_observed]["price"].values

print(f"Using LSOA {well_observed} with {len(population)} transactions")
print(f"Population Gini: {gini(population):.4f}")

# Repeatedly subsample at different sizes and measure Gini variance
sample_sizes = [10, 15, 20, 25, 30, 40, 50, 75, 100]
n_simulations = 1000

results = []
for n in sample_sizes:
    gini_values = [
        gini(np.random.choice(population, size=n, replace=False))
        for _ in range(n_simulations)
    ]
    results.append({
        "n": n,
        "mean_gini": np.mean(gini_values),
        "std_gini": np.std(gini_values),
        "cv": np.std(gini_values) / np.mean(gini_values)
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
# Calculate Gini for all well-observed LSOAs first
well_observed_lsoas = transactions_per_lsoa[transactions_per_lsoa > 200].index

lsoa_ginis = {
    lsoa: gini(land_reg_london[land_reg_london["lsoa21"] == lsoa]["price"].values)
    for lsoa in well_observed_lsoas
}

lsoa_ginis_series = pd.Series(lsoa_ginis).sort_values(ascending=False)
print(f"Top 10 highest Gini LSOAs:")
print(lsoa_ginis_series.head(10))

# Run stability simulation on highest Gini LSOA
high_gini_lsoa = lsoa_ginis_series.index[0]
population_high = land_reg_london[land_reg_london["lsoa21"] == high_gini_lsoa]["price"].values

print(f"\nUsing LSOA {high_gini_lsoa} with {len(population_high)} transactions")
print(f"Population Gini: {gini(population_high):.4f}")

results_high = []
for n in sample_sizes:
    gini_values = [
        gini(np.random.choice(population_high, size=n, replace=False))
        for _ in range(n_simulations)
    ]
    results_high.append({
        "n": n,
        "mean_gini": np.mean(gini_values),
        "std_gini": np.std(gini_values),
        "cv": np.std(gini_values) / np.mean(gini_values)
    })

results_high_df = pd.DataFrame(results_high)
print(results_high_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CV comparison
axes[0].plot(results_df["n"], results_df["cv"],
             marker="o", label=f"Low Gini LSOA ({gini(population):.3f})",
             color="steelblue")
axes[0].plot(results_high_df["n"], results_high_df["cv"],
             marker="o", label=f"High Gini LSOA ({gini(population_high):.3f})",
             color="darkorange")
axes[0].axvline(x=30, color="red", linestyle="--", label="n=30")
axes[0].axvline(x=50, color="green", linestyle="--", label="n=50")
axes[0].set_title("Gini estimate stability (CV) by sample size")
axes[0].set_xlabel("Number of transactions (n)")
axes[0].set_ylabel("Coefficient of variation")
axes[0].legend()

# Bias comparison
axes[1].plot(results_df["n"], results_df["mean_gini"],
             marker="o", label=f"Low Gini LSOA", color="steelblue")
axes[1].axhline(y=gini(population), color="steelblue",
                linestyle=":", label=f"True low Gini ({gini(population):.3f})")
axes[1].plot(results_high_df["n"], results_high_df["mean_gini"],
             marker="o", label=f"High Gini LSOA", color="darkorange")
axes[1].axhline(y=gini(population_high), color="darkorange",
                linestyle=":", label=f"True high Gini ({gini(population_high):.3f})")
axes[1].axvline(x=30, color="red", linestyle="--", label="n=30")
axes[1].axvline(x=50, color="green", linestyle="--", label="n=50")
axes[1].set_title("Gini estimate bias by sample size")
axes[1].set_xlabel("Number of transactions (n)")
axes[1].set_ylabel("Mean estimated Gini")
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUTS / "figures" / "gini_stability.png", dpi=150, bbox_inches="tight")
plt.show()

## Threshold decision: n=30

The stability simulation shows a meaningful elbow in the CV curve around n=30,
after which improvements in stability are more gradual. The bias analysis shows
the low-Gini LSOA converges to its true value by n=30, though the high-Gini
LSOA remains slightly downwardly biased even at n=50.

A threshold of 30 transactions is applied, balancing estimator reliability
against geographic coverage. This recovers the majority of sub-threshold LSOAs
in high-renter boroughs while remaining above the point where Gini estimates
become unreliable. Sensitivity analysis rerunning the regression models at n=50
is included in the modelling notebook to confirm findings are not
threshold-dependent.

Note: pandemic-affected years (2020-2021) were excluded due to stamp duty
holiday distortions inflating transaction volumes and compressing price
differentials. The 2022-2024 window represents a normalised post-pandemic market.

In [ ]:
# Calculate Gini per LSOA, exclude LSOAs below threshold
gini_by_lsoa = (land_reg_london.groupby("lsoa21")["price"]
                .apply(lambda x: gini(x) if len(x) >= 30 else np.nan)
                .reset_index()
                .rename(columns={"lsoa21": "LSOA21CD", "price": "gini_housing"}))

# Merge with full LSOA list to ensure all 4,994 LSOAs are represented
# LSOAs with no transactions will have NaN gini_housing
gini_by_lsoa = (london_lsoas[["LSOA21CD"]]
                .merge(gini_by_lsoa, on="LSOA21CD", how="left"))

print(f"LSOAs with Gini calculated: {gini_by_lsoa['gini_housing'].notna().sum()}")
print(f"LSOAs below threshold (NaN): {gini_by_lsoa['gini_housing'].isna().sum()}")
print(f"\nGini summary:")
print(gini_by_lsoa['gini_housing'].describe())

In [ ]:
# Load London boundaries
london_gdf = gpd.read_file(
    DATA_INTERIM / "boundaries" / "london_lsoa_2021.gpkg",
    layer="london_lsoa_2021"
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left — Gini distribution
axes[0].hist(gini_by_lsoa["gini_housing"].dropna(),
             bins="auto", edgecolor="black", color="steelblue")
axes[0].set_title("Distribution of housing Gini by LSOA (2022-2024)")
axes[0].set_xlabel("Gini coefficient")
axes[0].set_ylabel("Count of LSOAs")

# Right — map
london_gdf_gini = london_gdf.merge(gini_by_lsoa, on="LSOA21CD", how="left")

london_gdf_gini.plot(
    column="gini_housing",
    ax=axes[1],
    legend=True,
    cmap="YlOrRd",
    missing_kwds={"color": "lightgrey", "label": "Below threshold"}
)
axes[1].set_title("Housing Gini coefficient — London LSOAs (2022-2024)")
axes[1].set_axis_off()

plt.tight_layout()
plt.savefig(OUTPUTS / "figures" / "london_gini_distribution.png",
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Compare with paper's reported statistics
print("My Gini vs paper's Gini:")
print(f"Mean:  {gini_by_lsoa['gini_housing'].mean():.3f} vs 0.198 (paper)")
print(f"SD:    {gini_by_lsoa['gini_housing'].std():.3f} vs 0.086 (paper)")
print(f"Min:   {gini_by_lsoa['gini_housing'].min():.3f} vs 0.042 (paper)")
print(f"Max:   {gini_by_lsoa['gini_housing'].max():.3f} vs 0.649 (paper)")

In [ ]:
os.makedirs(DATA_INTERIM / "inequality", exist_ok=True)

gini_by_lsoa.to_csv(
    DATA_INTERIM / "inequality" / "gini_housing_2022_2024.csv",
    index=False
)

print(f"Saved {gini_by_lsoa['gini_housing'].notna().sum()} LSOA Gini values")